# AutoGen on Groq: The Basics [Agent Patterns - Module 13]

> **MLCourse - Agentic AI - Agent Patterns**

You have now built agents three ways in this course: **LangChain** chains,
**LangGraph** graphs, and **CrewAI** crews. This module adds a fourth,
**AutoGen** (Microsoft), and - more usefully - puts all of them side by side
on *the same task*, so the comparison is about the frameworks rather than
about four different problems.

AutoGen's organising idea is different from the others: agents are
**conversational participants**. You do not draw a graph or assign roles to a
process; you put agents in a chat and define when the chat should stop.

### What you will learn

1. Pointing AutoGen at Groq's OpenAI-compatible endpoint.
2. Why `model_info` is mandatory and what happens without it.
3. `AssistantAgent`, `run()`, and the `TaskResult` message list.
4. Streaming, and why AutoGen is async all the way down.
5. Multi-turn state on a single agent.

### Key takeaways

- No Groq-specific client exists or is needed - change `base_url`.
- AutoGen's API is **async**; that is not optional.
- The message list is the state. Read it, do not guess at it.

> **Note on scope.** The OpenAI Agents SDK is deliberately excluded from this
> course - it requires OpenAI as the provider, and this track is Groq-only.

### Setup: imports, environment, track discovery


In [ ]:
import os
import sys
import json
import time
import random
import asyncio
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

print(f"Track root : {TRACK}")
print(f"Model      : {MODEL} (via Groq's OpenAI-compatible endpoint)")


### Point AutoGen at Groq


In [ ]:
# AutoGen ships an OpenAI client. Groq exposes an OpenAI-COMPATIBLE endpoint,
# so we reuse that client and only change the base_url. This is the standard
# way to run AutoGen on a non-OpenAI provider - there is no Groq-specific
# client to install.
#
# `model_info` is REQUIRED for any model AutoGen does not have a built-in
# capability table for. Without it you get a ValueError before a single
# request goes out. You are telling the framework what the model can do.

from autogen_ext.models.openai import OpenAIChatCompletionClient

def make_client():
    return OpenAIChatCompletionClient(
        model=MODEL,
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1",   # <- the only Groq-specific line
        temperature=0.0,
        max_tokens=500,                              # free tier is 8000 TPM
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "family": "unknown",
            "structured_output": False,
        },
    )

print("make_client() ready")


### 1. The client

Three things in that cell deserve attention.

**`base_url`.** That single line is the entire Groq integration. Any provider
speaking the OpenAI wire format works the same way - Together, Fireworks, a
local vLLM server, Ollama's compatibility endpoint.

**`model_info`.** AutoGen keeps a capability table for models it knows. Groq's
`qwen/qwen3.8-27b` is not in it, so you must declare the capabilities
yourself. If you get them wrong - claiming `function_calling: True` for a
model that cannot - AutoGen will happily send tool definitions and fail at
runtime. Check your provider's docs; do not guess.

**`max_tokens`.** Set it. A conversational framework can loop, and a loop
without a token cap on a shared free tier is how you spend your whole
minute's budget on one cell.

### What happens without model_info


In [ ]:
# Worth seeing once so you recognise the error message later.

try:
    bad = OpenAIChatCompletionClient(
        model=MODEL,
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1",
    )
    print("no error raised at construction:", type(bad).__name__)
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:220]}")


### 2. A single agent

`AssistantAgent` is AutoGen's basic unit: a name, a model client, and a system
message. `run(task=...)` executes it and returns a `TaskResult`.

Note the `await`. AutoGen's public API is asynchronous throughout - in a
notebook you can `await` at the top level of a cell, and in a script you wrap
it in `asyncio.run()`. This is not a stylistic choice on their part: the whole
design assumes concurrent agents.

### The shared task


In [ ]:
# This module solves ONE task in three frameworks so the comparison is about
# the frameworks, not the problem. It is the same shape as the CrewAI crew in
# 04_crewai/01_fundamentals/05_research_assistant_crew: a writer produces a
# short piece from fixed reference notes, a critic reviews it, the writer
# revises. Deliberately tiny - we are studying plumbing, not prose.

NOTES = """Research notes: agent frameworks, 2026.
- Frameworks matured: CrewAI (role-based crews), LangGraph (explicit graphs),
  AutoGen (conversational agents).
- Agents are moving from demos to production.
- Main challenges: reliability, cost control, observability."""

TASK = ("Using ONLY the notes below, write a 2-sentence summary for an "
        "engineering newsletter.\n\n" + NOTES)

print(TASK)


### One agent, one task


In [ ]:
from autogen_agentchat.agents import AssistantAgent

client = make_client()

writer = AssistantAgent(
    name="writer",
    model_client=client,
    system_message=("You write concise engineering newsletter copy. "
                    "Use only the facts you are given. No bullet points."),
)

t0 = time.time()
result = await writer.run(task=TASK)
elapsed = time.time() - t0

print(f"stop_reason : {result.stop_reason}")
print(f"messages    : {len(result.messages)}")
print(f"elapsed     : {elapsed:.1f}s")
print()
print(result.messages[-1].content)


### The TaskResult is a message list


In [ ]:
# This is AutoGen's state. Everything the framework knows is in here.

for i, m in enumerate(result.messages):
    kind = type(m).__name__
    src = getattr(m, "source", "-")
    body = str(getattr(m, "content", ""))
    usage = getattr(m, "models_usage", None)
    tok = f" tokens={usage.prompt_tokens}+{usage.completion_tokens}" if usage else ""
    print(f"[{i}] {kind:18s} source={src:8s}{tok}")
    print(f"     {body[:150]}")


Two things to notice:

- The **task itself** is message 0, as a `TextMessage` from `user`. AutoGen
  does not distinguish "the task" from "the conversation" - the task *is* the
  first message.
- `models_usage` carries per-message token counts. That is your cost trail,
  and on a shared free tier you should be reading it.

### 3. Streaming

`run_stream()` yields messages as they are produced instead of returning at
the end. In a chat UI this is how you show progress; in a multi-agent team it
is how you watch which agent is speaking without waiting for the whole
conversation.

### Streaming the same call


In [ ]:
from autogen_agentchat.base import TaskResult

async for item in writer.run_stream(task="In one sentence: what is an agent framework for?"):
    if isinstance(item, TaskResult):
        print(f"\n[done] stop_reason={item.stop_reason}")
    else:
        print(f"[{getattr(item, 'source', '?')}] {str(getattr(item, 'content', ''))[:200]}")


Note the shape of that loop: the stream yields **messages**, and then finally
one `TaskResult`. You must type-check the item. Forgetting to is the most
common AutoGen streaming bug - you end up calling `.content` on the result
object and get an `AttributeError` at the very end of an otherwise working
run.

### 4. Agent memory

An `AssistantAgent` keeps its message history between `run()` calls by
default. That is convenient and it is also a trap: the history grows, every
call re-sends it, and your token bill grows quadratically over a long session.

`on_reset()` clears it. Decide explicitly which behaviour you want.

### State carries across calls


In [ ]:
memo = AssistantAgent(
    name="memo",
    model_client=client,
    system_message="Answer in at most 8 words.",
)

r1 = await memo.run(task="My deployment target is Kubernetes. Acknowledge briefly.")
print("turn 1:", r1.messages[-1].content)

r2 = await memo.run(task="What did I say my deployment target was?")
print("turn 2:", r2.messages[-1].content)

await memo.on_reset(cancellation_token=None)
r3 = await memo.run(task="What did I say my deployment target was?")
print("after reset:", r3.messages[-1].content)


### Always close the client


In [ ]:
# It holds an HTTP session. Leaking it in a notebook leaves open connections.

await client.close()
print("client closed")


### Pitfalls recap

- **Missing `model_info`.** Required for any model AutoGen does not know.
  Declare capabilities honestly.
- **Forgetting `await`.** The API is async everywhere. A forgotten `await`
  gets you a coroutine object, not a result.
- **Not type-checking in `run_stream()`.** The last yielded item is a
  `TaskResult`, not a message.
- **Unbounded agent history.** State persists across `run()` calls. Reset it
  or pay for it.
- **No `max_tokens`.** Conversational frameworks loop; caps are your brake.
- **Leaking the client.** `await client.close()`.

### Next

Notebook 02 puts two agents in a conversation and - crucially - defines when
that conversation stops.